# PMM Dynamic Multi-Exchange Sweep

**Automated optimization across multiple exchanges with exchange-separated results**

This notebook:
1. Discovers all available pairs for multiple connectors + one quote asset from MongoDB
2. For each eligible connector / pair:
   - Runs Optuna walk-forward optimization
   - Stress-tests the top candidates
   - Evaluates the best stress-validated candidate
3. Exports YAML configs and reports under `artifacts/sweep/<connector>/`
4. Displays summary tables separated by exchange

**Configuration:** Edit the variables in the first code cell, then Run All.


In [1]:
import sys, os, subprocess, time, logging
from datetime import datetime, timezone

PMM_DIR = "/quants-lab/research_notebooks/market_lab/pmm_dynamic"
if PMM_DIR not in sys.path:
    sys.path.insert(0, PMM_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", PMM_DIR, "--quiet"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import numpy as np
import pandas as pd
import optuna
import pmm_lab

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

print(f"pmm_lab {pmm_lab.__version__} | NumPy {np.__version__} | Optuna {optuna.__version__}")

MONGO_URI = os.getenv("MONGO_URI", "")
OPTUNA_STORAGE = os.getenv("OPTUNA_STORAGE", "")
print(f"MONGO_URI      : {'SET' if MONGO_URI else 'NOT SET'}")
print(f"OPTUNA_STORAGE : {'SET' if OPTUNA_STORAGE else 'NOT SET (using SQLite)'}")
from pmm_lab.optuna.preflight import print_environment, run_preflight
print_environment()

pmm_lab 0.1.0 | NumPy 2.2.6 | Optuna 4.7.0
MONGO_URI      : SET
OPTUNA_STORAGE : SET
Python     : 3.12.13
NumPy      : 2.2.6
Pandas     : 3.0.1
Optuna     : 4.7.0
pmm_lab    : 0.1.0
Storage    : PostgreSQL (SET)
CPU cores  : 32
OMP_NUM_THREADS          : 1
OPENBLAS_NUM_THREADS     : 1
MKL_NUM_THREADS          : 1
NUMEXPR_NUM_THREADS      : 1


## 1. Configuration

Edit these variables to control the multi-exchange sweep. Then **Run All** cells below.


In [2]:
# ==============================================================
# SWEEP CONFIGURATION — edit these, then Run All
# ==============================================================
# Use this as the operating rule:
# 3000–5000: coarse screening / pair triage
# 8000–10000: good default for a serious cross-exchange search in this notebook
# 12000–15000: only for finalists or very noisy pairs
# ==============================================================

CONNECTORS = ["mexc", "nonkyc"]  # Exchanges to sweep together
QUOTE_ASSET = "*"             # Quote asset filter (pairs ending in -USDT)
N_TRIALS = 12000                 # Optuna trials per connector / pair
PERC_TRIALS_TEST = .05           # what percentage of the N_TRIALS should be completely random
TOP_N = 75                       # Top candidates to stress test
MIN_ROBUST_SCORE = -5.0           # Minimum robust score to export (0 = breakeven) - changed to -5 for testing
N_JOBS = 8                       # Parallel Optuna workers

# Preferred interval per connector (add your own)
CONNECTOR_INTERVALS = {
    "nonkyc": "5m",
    "mexc": "5m",
}
DEFAULT_INTERVAL = "5m"

# Minimum data requirement (days)
MIN_DATA_DAYS = 56

# Maximum training window (days). Only the most recent N days of candle
# data will be used for walk-forward optimization. Set to None to use all
# available data (original behaviour).
MAX_TRAINING_DAYS = 180

# Feature computation mode for search AND stress/validation.
# False = fast vectorized (for broad search), True = controller-equivalent sliding window.
# NOTE: stress/validation uses the same controller_compat setting as search.
SEARCH_CONTROLLER_COMPAT = False

# Validation controller mode — True = controller-equivalent sliding window for finalist
# evaluation (holdout, recent-window, sensitivity). This is intentional: search is fast,
# validation is realistic.
VALIDATION_CONTROLLER_COMPAT = True

# Stale data gate — skip pairs whose most recent candle is older than this
MAX_STALE_DAYS = 7

# Phase-1 minimum score to proceed to stress testing
# If best phase-1 score <= this, skip stress (saves compute on clearly bad pairs)
MIN_PHASE1_BEST_FOR_STRESS = -0.5

OBJECTIVE_VERSION = 2

# ==============================================================

CONNECTORS = [c.strip().lower() for c in CONNECTORS]
INTERVALS_BY_CONNECTOR = {
    connector: CONNECTOR_INTERVALS.get(connector, DEFAULT_INTERVAL)
    for connector in CONNECTORS
}

from pmm_lab.config.defaults import INTERVAL_SECONDS

print(f"Connectors     : {', '.join(CONNECTORS)}")
print(f"Quote asset    : {QUOTE_ASSET}")
print(f"Intervals      : {', '.join(f'{c}:{INTERVALS_BY_CONNECTOR[c]}' for c in CONNECTORS)}")
print(f"Trials/pair    : {N_TRIALS}")
print(f"Top-N stress   : {TOP_N}")
print(f"Min score      : {MIN_ROBUST_SCORE}")
print(f"Min data days  : {MIN_DATA_DAYS}")
print(f"Search mode    : controller_compat={SEARCH_CONTROLLER_COMPAT}")
print(f"Max stale days : {MAX_STALE_DAYS}")
print(f"Max training   : {MAX_TRAINING_DAYS}d" if MAX_TRAINING_DAYS else "Max training   : unlimited")


Connectors     : mexc, nonkyc
Quote asset    : *
Intervals      : mexc:5m, nonkyc:5m
Trials/pair    : 12000
Top-N stress   : 75
Min score      : -5.0
Min data days  : 56
Search mode    : controller_compat=False
Max stale days : 7
Max training   : 180d


In [3]:
# ── Preflight: validate storage + worker configuration ──
# The optimize_study_for_notebook() helper handles dispatch (serial vs
# process-parallel) internally, including SQLite fallback and preflight
# checks. This cell only prints environment info for operator visibility.
from pmm_lab.optuna.preflight import run_preflight
from pmm_lab.optuna.storage import get_storage_url

_storage_url = OPTUNA_STORAGE if OPTUNA_STORAGE else get_storage_url()
_is_postgres = "postgresql" in str(_storage_url).lower()

try:
    preflight_report = run_preflight(
        n_workers=N_JOBS,
        storage_url=_storage_url,
        strict=False,
    )
except Exception as e:
    print(f"Preflight info: {e}")

print(f"Requested N_JOBS: {N_JOBS}")
print(f"Storage backend : {'PostgreSQL' if _is_postgres else 'SQLite (fallback)'}")
print(f"Dispatch mode   : {'process-parallel (if preflight passes)' if N_JOBS > 1 and _is_postgres else 'serial'}")

Preflight: ALL CHECKS PASSED
Requested N_JOBS: 8
Storage backend : PostgreSQL
Dispatch mode   : process-parallel (if preflight passes)


## 2. Discover Available Pairs Across Exchanges

In [4]:
from pmm_lab.data.mongo import MongoCandleLoader
from pmm_lab.config.params import DataQuery
from pmm_lab.data.hashing import hash_candles
from datetime import datetime, timezone

loader = MongoCandleLoader()
all_combos = loader.list_combos(connector=None, quote_asset=QUOTE_ASSET)

now_ts = datetime.now(timezone.utc).timestamp()


# Filter to our selected connectors, per-connector interval, and minimum data
candidates = []
stale_exclusions = []
insufficient_exclusions = []

for combo in all_combos:
    connector = combo["connector"]
    if connector not in CONNECTORS:
        continue

    interval = INTERVALS_BY_CONNECTOR.get(connector, DEFAULT_INTERVAL)
    if combo["interval"] != interval:
        continue

    # Cap effective start to training window
    effective_first_ts = combo["first_ts"]
    if MAX_TRAINING_DAYS is not None:
        training_cutoff_ts = combo["last_ts"] - (MAX_TRAINING_DAYS * 86400)
        effective_first_ts = max(effective_first_ts, training_cutoff_ts)
    data_days = (combo["last_ts"] - effective_first_ts) / 86400

    if data_days < MIN_DATA_DAYS:
        insufficient_exclusions.append({
            "connector": connector,
            "trading_pair": combo["trading_pair"],
            "count": combo["count"],
            "interval": interval,
            "data_days": data_days,
            "reason": f"< {MIN_DATA_DAYS}d data",
        })
        continue

    # Stale-pair gate: check recency of last candle
    last_age_days = (now_ts - combo["last_ts"]) / 86400
    if last_age_days > MAX_STALE_DAYS:
        last_utc = datetime.fromtimestamp(combo["last_ts"], tz=timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
        stale_exclusions.append({
            "connector": connector,
            "trading_pair": combo["trading_pair"],
            "count": combo["count"],
            "interval": interval,
            "data_days": data_days,
            "last_age_days": last_age_days,
            "last_utc": last_utc,
            "reason": f"stale ({last_age_days:.1f}d old > {MAX_STALE_DAYS}d)",
        })
        continue

    candidates.append({
        "connector": connector,
        "trading_pair": combo["trading_pair"],
        "interval": interval,
        "count": combo["count"],
        "first_ts": effective_first_ts,
        "full_first_ts": combo["first_ts"],
        "last_ts": combo["last_ts"],
        "data_days": data_days,
    })

candidates = sorted(candidates, key=lambda c: (c["connector"], c["trading_pair"]))

print(f"\n{'='*60}")
print(f"Found {len(candidates)} connector/pair combinations with >= {MIN_DATA_DAYS} days of data")
print(f"{'='*60}")

for connector in CONNECTORS:
    connector_candidates = [c for c in candidates if c["connector"] == connector]
    interval = INTERVALS_BY_CONNECTOR.get(connector, DEFAULT_INTERVAL)
    print(f"\n{connector} / {QUOTE_ASSET} / {interval}: {len(connector_candidates)} pair(s)")
    if connector_candidates:
        for c in connector_candidates:
            print(f"  {c['trading_pair']:15s}  {c['count']:>8,} candles  {c['data_days']:5.1f} days")
    else:
        print("  (no eligible pairs found)")

if stale_exclusions:
    print(f"\nExcluded {len(stale_exclusions)} stale pair(s) (last candle > {MAX_STALE_DAYS}d old):")
    for ex in stale_exclusions:
        print(f"  {ex['connector']:8s}  {ex['trading_pair']:15s}  last={ex['last_utc']}  age={ex['last_age_days']:.1f}d")

if insufficient_exclusions:
    print(f"\nExcluded {len(insufficient_exclusions)} pair(s) with insufficient data:")
    for ex in insufficient_exclusions:
        print(f"  {ex['connector']:8s}  {ex['trading_pair']:15s}  {ex['data_days']:.1f} days")

print(f"\nTotal connector/pair combinations to optimize: {len(candidates)}")



Found 56 connector/pair combinations with >= 56 days of data

mexc / * / 5m: 31 pair(s)
  ADA-USDT           53,020 candles  180.0 days
  APT-USDT           57,828 candles  180.0 days
  ASTER-USDT         53,020 candles  180.0 days
  ATOM-USDT          53,020 candles  180.0 days
  BNB-USDT           57,836 candles  180.0 days
  BTC-USDT           59,096 candles  180.0 days
  DOGE-USDT          59,081 candles  180.0 days
  DOT-USDT           53,020 candles  180.0 days
  ETH-USDT           59,090 candles  180.0 days
  HYPE-USDT          53,021 candles  180.0 days
  ICP-USDT           53,022 candles  180.0 days
  LTC-USDT           53,020 candles  180.0 days
  OP-USDT            53,020 candles  180.0 days
  PEPE-USDT          57,824 candles  180.0 days
  PUMP-USDT          53,020 candles  180.0 days
  RENDER-USDT        53,020 candles  180.0 days
  SAHARA-USDT        51,946 candles  180.0 days
  SAL-USDT           59,068 candles  180.0 days
  SHIB-USDT          53,020 candles  180.0 days

## 3. Sweep: Optimize Each Connector / Pair

For each eligible connector / pair, the sweep:
1. Loads and validates candles
2. Auto-scales walk-forward windows to fit available data
3. Runs Optuna trials (walk-forward, stress OFF)
4. Stress-tests the top candidates
5. Records the best stress-validated result


In [ ]:
# ── Config guard: ensure configuration cell was executed ──
_required_config = [
    "VALIDATION_CONTROLLER_COMPAT", "SEARCH_CONTROLLER_COMPAT",
    "OBJECTIVE_VERSION", "N_TRIALS", "TOP_N", "MIN_ROBUST_SCORE",
    "N_JOBS", "MIN_PHASE1_BEST_FOR_STRESS",
]
_missing = [v for v in _required_config if v not in globals()]
if _missing:
    import warnings as _w
    _w.warn(
        f"Configuration cell may not have been executed. "
        f"Missing: {', '.join(_missing)}. "
        f"Applying safe defaults — re-run all cells from the top for your custom settings.",
        stacklevel=1,
    )
    # Safe defaults so the sweep can still proceed
    if "VALIDATION_CONTROLLER_COMPAT" not in globals():
        VALIDATION_CONTROLLER_COMPAT = True
    if "SEARCH_CONTROLLER_COMPAT" not in globals():
        SEARCH_CONTROLLER_COMPAT = False
    if "OBJECTIVE_VERSION" not in globals():
        OBJECTIVE_VERSION = 2

from pmm_lab.data.candles import validate_candles
from pmm_lab.config.exchange_rules import load_exchange_rules, resolve_pair_rules
from pmm_lab.optuna.notebook_dispatch import optimize_study_for_notebook
from pmm_lab.optuna.objective_wrapper import create_objective
from pmm_lab.optuna.callbacks import DegeneracyCheckCallback, TrialLoggingCallback
from pmm_lab.optuna.canonicalizer import canonicalize_params
from pmm_lab.objective.stress import load_stress_scenarios
from pmm_lab.objective.stress_selection import select_best_stressed_candidate
from pmm_lab.objective.walkforward import run_walk_forward
from pmm_lab.objective.objective import REJECT_SCORE, objective_v1
from pmm_lab.export.hb_yaml import export_yaml, ExportParams
from pmm_lab.export.validate_export import validate_yaml_file
from pmm_lab.report.report_md import generate_report, run_stop_ship_checks
from pmm_lab.sim.runner import CandleSimRunner
from pmm_lab.objective.recent_window import evaluate_recent_window
from pmm_lab.objective.holdout import evaluate_holdout
from pmm_lab.objective.dataset_split import split_for_release_gate
from pmm_lab.optuna.sensitivity import compute_sensitivity
from pmm_lab.optuna.clustering import analyze_top_k
from pmm_lab.parity.feature_parity import check_feature_parity_frozen
from pmm_lab.parity.fixtures import load_frozen_fixture
from dataclasses import replace as _replace

# Preload stress scenarios once (Task 4.1)
stress_scenarios = load_stress_scenarios()

rules_db = load_exchange_rules()
sweep_results = []
sweep_start = time.time()

for pair_idx, pair_info in enumerate(candidates):
    connector = pair_info["connector"]
    pair = pair_info["trading_pair"]
    interval = pair_info["interval"]
    bar_interval_seconds = INTERVAL_SECONDS[interval]

    print(f"\n{'═'*60}")
    print(f"  [{pair_idx+1}/{len(candidates)}] {connector} / {pair} / {interval}")
    print(f"{'═'*60}")

    pair_start = time.time()

    # ── Load candles ──
    try:
        _start_ts = int(pair_info["first_ts"]) if MAX_TRAINING_DAYS is not None else None
        query = DataQuery(connector=connector, trading_pair=pair, interval=interval, start_ts=_start_ts)
        candles = loader.load_range(query)
        audit = validate_candles(candles, interval=interval, strict=True)
        if not audit.passed_strict:
            print(f"  SKIP: audit failed — {audit.failure_reasons}")
            sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                                  "status": "audit_fail", "robust_score": None})
            continue
        dataset_hash = hash_candles(candles)
    except Exception as e:
        print(f"  SKIP: load failed — {e}")
        sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                              "status": "load_fail", "robust_score": None})
        continue


    # ── Dataset split for release gate ──
    try:
        dataset_slices = split_for_release_gate(candles, recent_days=28, holdout_fraction=0.20, min_pre_release_bars=200, min_holdout_bars=50)
        dev_candles = dataset_slices.dev_candles
        dev_dataset_hash = hash_candles(dev_candles)
        print(f"  Split: dev={len(dev_candles)} holdout={len(dataset_slices.holdout_candles)} recent={len(dataset_slices.recent_release_candles)}")
    except ValueError as e:
        print(f"  Split failed ({e}), using full candles")
        dataset_slices = None
        dev_candles = candles
        dev_dataset_hash = dataset_hash

    # ── Exchange rules ──
    try:
        pair_rules = resolve_pair_rules(rules_db, connector, pair)
    except KeyError:
        # Fall back to connector defaults if pair-specific rules not found
        try:
            pair_rules = resolve_pair_rules(rules_db, connector, "DEFAULT")
        except KeyError:
            print(f"  SKIP: no exchange rules for {connector}/{pair}")
            sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                                  "status": "no_rules", "robust_score": None})
            continue

    ref_price = float(np.median(candles["close"]))

    # ── Auto-scale walk-forward windows ──
    dataset_days = len(candles) * bar_interval_seconds / 86400
    if dataset_days >= 120:
        train_days, test_days, step_days = 42.0, 14.0, 14.0
    elif dataset_days >= 60:
        train_days, test_days, step_days = 21.0, 7.0, 7.0
    elif dataset_days >= 28:
        train_days, test_days, step_days = 10.0, 4.0, 4.0
    else:
        print(f"  SKIP: only {dataset_days:.1f} days of data")
        sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                              "status": "insufficient_data", "robust_score": None})
        continue

    print(f"  Candles: {len(candles):,}  Days: {dataset_days:.1f}  "
          f"WF: {train_days}/{test_days}/{step_days}d  Ref: {ref_price:,.4f}")

    if MAX_TRAINING_DAYS is not None and pair_info.get("full_first_ts"):
        full_days = (pair_info["last_ts"] - pair_info["full_first_ts"]) / 86400
        used_days = (pair_info["last_ts"] - pair_info["first_ts"]) / 86400
        if full_days > used_days + 1:
            print(f"  Training window: {used_days:.0f}d of {full_days:.0f}d available (capped to {MAX_TRAINING_DAYS}d)")
    
    # ── Phase 1: Optimization ──
    study_name = f"{connector}_{pair}_{interval}_sweep_v1"

    try:
        study = optimize_study_for_notebook(
            study_name=study_name,
            storage_url=OPTUNA_STORAGE if OPTUNA_STORAGE else None,
            n_trials=N_TRIALS,
            n_jobs=N_JOBS,
            objective_factory=create_objective,
            factory_kwargs=dict(
                candles=dev_candles,
                pair_rules=pair_rules,
                bar_interval_seconds=bar_interval_seconds,
                dataset_hash=dev_dataset_hash,
                reference_price=ref_price,
                train_days=train_days,
                test_days=test_days,
                step_days=step_days,
                run_stress=False,
                controller_compat=SEARCH_CONTROLLER_COMPAT,
                objective_version=OBJECTIVE_VERSION,
            ),
            callbacks=[DegeneracyCheckCallback()],
            n_startup_trials=int(N_TRIALS * PERC_TRIALS_TEST),
        )

        completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
        ranked = sorted(completed, key=lambda t: t.value, reverse=True)

        if not ranked:
            print(f"  Phase 1: {len(completed)} complete, {len(pruned)} pruned — NO COMPLETED TRIALS")
            sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                                  "status": "no_completed_trials", "robust_score": None})
            continue

        best_val = ranked[0].value
        print(f"  Phase 1: {len(completed)} complete, {len(pruned)} pruned, best={best_val:.4f}")
    except Exception as e:
        print(f"  SKIP: optimization failed — {e}")
        sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                              "status": "optim_fail", "robust_score": None})
        continue

    # ── Phase 1 score gate ──
    if best_val <= MIN_PHASE1_BEST_FOR_STRESS:
        print(f"  SKIP STRESS: phase-1 best ({best_val:.4f}) <= {MIN_PHASE1_BEST_FOR_STRESS}")
        sweep_results.append({
            "connector": connector,
            "pair": pair,
            "interval": interval,
            "status": "phase1_below_threshold",
            "robust_score": best_val,
            "phase1_best": best_val,
        })
        continue

    # ── Phase 2: Stress top N (with signal cache, dedup, early pruning) ──
    try:
        top_trials = ranked[:min(TOP_N, len(ranked))]

        top_candidates = []
        for trial in top_trials:
            config, reject = canonicalize_params(trial.params, pair_rules, ref_price)
            if config is not None:
                top_candidates.append({
                    "trial_number": trial.number,
                    "phase1_score": trial.value,
                    "params": trial.params,
                    "config": config,
                })

        if not top_candidates:
            print(f"  SKIP: no valid configs to stress test")
            sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                                  "status": "no_valid_configs", "robust_score": None})
            continue

        # Deduplicate by full config fingerprint (Task 4.4)
        seen_configs = {}
        deduped_candidates = []
        for candidate in top_candidates:
            fingerprint = candidate["config"].to_fingerprint()
            if fingerprint not in seen_configs:
                seen_configs[fingerprint] = True
                deduped_candidates.append(candidate)
        print(f"  Deduped: {len(top_candidates)} -> {len(deduped_candidates)} unique configs")
        top_candidates = deduped_candidates

        # Signal cache + early pruning (Tasks 4.3, 4.5)
        signal_cache = {}
        best, diag = select_best_stressed_candidate(
            top_candidates, dev_candles, pair_rules, bar_interval_seconds,
            scenarios=stress_scenarios,
            signal_cache=signal_cache,
            objective_version=OBJECTIVE_VERSION,
        )

        if best is None:
            print(f"  SKIP: no candidates survived stress testing")
            sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                                  "status": "stress_fail", "robust_score": None})
            continue

        best_config = best["config"]
        best_stress = best["stress_report"]
        # Reuse winner baseline metrics (Task 4.2) — no extra sim needed
        bm = best_stress.baseline_metrics

        pair_elapsed = time.time() - pair_start
        print(f"  Best: trial {best['trial_number']}  robust={best['robust_score']:.4f}  "
              f"PnL={bm.pnl_pct:.2f}%  trades={bm.trade_count}  ({pair_elapsed/60:.1f}min)")
        print(f"  Stress diag: evaluated={diag['candidates_evaluated']} "
              f"pruned={diag['candidates_pruned']} "
              f"cache_hits={diag['signal_cache_hits']} misses={diag['signal_cache_misses']}")

    except Exception as e:
        print(f"  SKIP: stress testing failed — {e}")
        sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                              "status": "stress_fail", "robust_score": None})
        continue


    # ── Finalist validation ──
    val_config = _replace(best_config, controller_compat=VALIDATION_CONTROLLER_COMPAT)

    recent_window_result = None
    try:
        recent_window_result = evaluate_recent_window(
            full_candles=candles, config=val_config, pair_rules=pair_rules,
            bar_interval_seconds=bar_interval_seconds,
            recent_days=28, run_stress=True, objective_version=OBJECTIVE_VERSION,
        )
        print(f"  Recent 28d: {'PASS' if recent_window_result.passed else 'FAIL'} \u2014 {recent_window_result.reason}")
    except Exception as e:
        print(f"  Recent 28d: ERROR \u2014 {e}")

    holdout_report = None
    try:
        if dataset_slices is not None:
            holdout_candles_h = dataset_slices.holdout_candles
            holdout_start_idx = dataset_slices.holdout_start_idx_in_pre_release
        else:
            from pmm_lab.objective.holdout import split_holdout
            dev_candles_h, holdout_candles_h = split_holdout(candles, 0.20, min_holdout_bars=50)
            holdout_start_idx = len(dev_candles_h)
        holdout_candidates = [(val_config, best.get("robust_score", 0.0))]
        for t_idx in range(1, min(5, len(top_candidates))):
            tc = top_candidates[t_idx]
            tc_config = canonicalize_params(tc["params"], pair_rules, ref_price)[0]
            if tc_config is not None:
                tc_config = _replace(tc_config, controller_compat=VALIDATION_CONTROLLER_COMPAT)
                holdout_candidates.append((tc_config, tc.get("phase1_score", 0.0)))
        holdout_report = evaluate_holdout(
            holdout_candles_h, holdout_candidates, pair_rules, bar_interval_seconds,
            run_stress=True, objective_version=OBJECTIVE_VERSION,
            full_candles=candles, holdout_start_idx=holdout_start_idx,
        )
        print(f"  Holdout: {'PASS' if holdout_report.exported_holdout_passed else 'FAIL'}")
    except Exception as e:
        print(f"  Holdout: ERROR \u2014 {e}")

    sensitivity_report = None
    sensitivity_penalty = None
    try:
        sensitivity_report = compute_sensitivity(
            best["params"], candles, pair_rules, bar_interval_seconds, ref_price,
            objective_version=OBJECTIVE_VERSION, controller_compat=VALIDATION_CONTROLLER_COMPAT,
        )
        sensitivity_penalty = sensitivity_report.sensitivity_penalty
        print(f"  Sensitivity: penalty={sensitivity_penalty:.4f}")
    except Exception as e:
        print(f"  Sensitivity: ERROR \u2014 {e}")

    cluster_report = None
    try:
        cluster_report = analyze_top_k(study, k=min(10, len(ranked)))
        print(f"  Clustering: {'CLUSTERED' if cluster_report.is_clustered else 'SCATTERED'}")
    except Exception as e:
        print(f"  Clustering: ERROR \u2014 {e}")

    parity_result = None
    long_parity_result = None
    try:
        from pathlib import Path as _Path
        _fix_base = _Path(__file__).resolve().parent.parent if '__file__' in dir() else _Path("fixtures")
        if not _fix_base.is_dir():
            _fix_base = _Path("research_notebooks/market_lab/pmm_dynamic/fixtures")
        if not _fix_base.is_dir():
            _fix_base = _Path("fixtures")
        _short = _fix_base / "short_100bar_compat"
        if _short.is_dir():
            _f = load_frozen_fixture(str(_short))
            parity_result = check_feature_parity_frozen(_f.candles, _f.expected_features, _f.config_params)
        _long = _fix_base / "long_500bar_compat"
        if _long.is_dir():
            _lf = load_frozen_fixture(str(_long))
            long_parity_result = check_feature_parity_frozen(_lf.candles, _lf.expected_features, _lf.config_params)
        print(f"  Parity: short={'PASS' if parity_result and parity_result.passed else 'N/A'}, long={'PASS' if long_parity_result and long_parity_result.passed else 'N/A'}")
    except Exception as e:
        print(f"  Parity: ERROR \u2014 {e}")

    full_validation_executed = all([recent_window_result is not None, holdout_report is not None])

    # ── Record result ──
    best_metrics = bm
    best_obj = best_stress.baseline_objective
    result_entry = {
        "connector": connector,
        "pair": pair,
        "interval": interval,
        "status": "complete",
        "robust_score": best["robust_score"],
        "baseline_score": best["baseline_score"],
        "worst_score": best["worst_score"],
        "worst_scenario": best["worst_scenario"],
        "pnl_pct": bm.pnl_pct,
        "sharpe": bm.sharpe,
        "max_dd_pct": bm.max_drawdown_pct,
        "trade_count": bm.trade_count,
        "total_fees": bm.total_fees_quote,
        "profit_factor": bm.profit_factor,
        "trial_number": best["trial_number"],
        "best_config": best_config,
        "best_params": best["params"],
        "best_stress": best_stress,
        "dataset_hash": dataset_hash,
        "n_candles": len(candles),
        "dataset_days": dataset_days,
        "train_days": train_days,
        "test_days": test_days,
        "step_days": step_days,
        "study_name": study_name,
        "recent_window_result": recent_window_result,
        "holdout_report": holdout_report,
        "sensitivity_report": sensitivity_report,
        "sensitivity_penalty": sensitivity_penalty,
        "cluster_report": cluster_report,
        "parity_result": parity_result,
        "long_parity_result": long_parity_result,
        "full_validation_executed": full_validation_executed,
        "dataset_slices": dataset_slices if 'dataset_slices' in dir() else None,
    }
    sweep_results.append(result_entry)

    # ── Export if profitable ──
    if best["robust_score"] >= MIN_ROBUST_SCORE:
        validation_result = None
        try:
            export_params = ExportParams(
                connector_name=connector,
                trading_pair=pair,
                candles_connector=connector,
                candles_trading_pair=pair,
                interval=interval,
            )

            yaml_path = export_yaml(
                config=best_config,
                output_path=f"artifacts/sweep/{connector}/{pair}_{interval}_screening_best.yaml",
                export_params=export_params,
                metadata={
                    "dataset_hash": dataset_hash,
                    "trial": best["trial_number"],
                    "phase1_score": best["phase1_score"],
                    "robust_score": best["robust_score"],
                    "worst_scenario": best["worst_scenario"],
                    "worst_score": best["worst_score"],
                    "sweep_date": datetime.now(timezone.utc).isoformat(),
                },
            )
            validation_result = validate_yaml_file(yaml_path)

            # Walk-forward for report
            wf_result = run_walk_forward(
                candles=candles, config=val_config, pair_rules=pair_rules,
                bar_interval_seconds=bar_interval_seconds, dataset_hash=dataset_hash,
                train_days=train_days, test_days=test_days, step_days=step_days,
                objective_version=OBJECTIVE_VERSION,
            )

            checks = run_stop_ship_checks(
                best_metrics=best_metrics, best_objective=best_obj,
                walkforward_result=wf_result, stress_report=best_stress,
                dataset_audit=audit,
                validation_result=validation_result,
                holdout_report=holdout_report,
                sensitivity_penalty=sensitivity_penalty,
                recent_window_result=recent_window_result,
                parity_result=parity_result,
                cluster_report=cluster_report,
                long_parity_result=long_parity_result,
            )

            _run_provenance = {
                "notebook": os.path.basename(__file__) if '__file__' in dir() else "jupyter",
                "run_timestamp": datetime.now(timezone.utc).isoformat(),
                "n_jobs": N_JOBS,
                "objective_version": OBJECTIVE_VERSION,
                "search_controller_compat": SEARCH_CONTROLLER_COMPAT,
                "validation_controller_compat": VALIDATION_CONTROLLER_COMPAT,
                "trial_number": best["trial_number"],
            }

            generate_report(
                study_name=study_name,
                dataset_summary={
                    "connector": connector, "trading_pair": pair, "interval": interval,
                    "n_candles": len(candles), "dataset_hash": dataset_hash,
                    "n_trials_phase1": N_TRIALS, "n_candidates_stressed": len(top_candidates),
                    "total_amount_quote_search_min": 25.0,
                    "total_amount_quote_search_max": 1000.0,
                    "total_amount_quote_ideal": best_config.total_amount_quote,
                    "search_controller_compat": SEARCH_CONTROLLER_COMPAT,
                },
                best_params=best["params"], best_metrics=best_metrics, best_objective=best_obj,
                walkforward_result=wf_result, stress_report=best_stress,
                stop_ship_checks=checks,
                holdout_report=holdout_report,
                dataset_audit=audit,
                sensitivity_report=sensitivity_report,
                recent_window_result=recent_window_result,
                cluster_report=cluster_report,
                yaml_validation_result=validation_result,
                dataset_slices=dataset_slices,
                parity_result=parity_result,
                long_parity_result=long_parity_result,
                run_provenance=_run_provenance,
                output_path=f"artifacts/sweep/{connector}/{pair}_{interval}_report.md",
            )

            total_time = time.time() - pair_start
            print(f"  Total time: ({pair_elapsed/60:.1f}min)")
            
            result_entry["exported"] = True
            result_entry["yaml_path"] = yaml_path
            all_pass = all(checks.values())
            if all_pass:
                import shutil
                _validated_path = yaml_path.replace("_screening_best.yaml", "_validated_best.yaml")
                shutil.copy2(yaml_path, _validated_path)
                print(f"  VALIDATED  yaml={_validated_path}")
            result_entry["all_checks_pass"] = all_pass
            print(f"  EXPORTED  yaml={yaml_path}  checks={'ALL PASS' if all_pass else 'SOME FAIL'}")
        except Exception as e:
            print(f"  Export failed: {e}")
            result_entry["exported"] = False
    else:
        result_entry["exported"] = False
        print(f"  NOT PROFITABLE (robust={best['robust_score']:.4f} < {MIN_ROBUST_SCORE})")

total_elapsed = time.time() - sweep_start
print(f"\n{'═'*60}")
print(f"SWEEP COMPLETE: {len(candidates)} connector/pair combinations in {total_elapsed/60:.1f} minutes")
print(f"{'═'*60}")





════════════════════════════════════════════════════════════
  [1/56] mexc / ADA-USDT / 5m
════════════════════════════════════════════════════════════
  Split: dev=35021 holdout=8755 recent=8065
  Candles: 51,841  Days: 180.0  WF: 42.0/14.0/14.0d  Ref: 0.3896
  Training window: 180d of 184d available (capped to 180d)


Callbacks are not supported in parallel mode. They will be skipped. Use post-hoc analysis instead.


Preflight: ALL CHECKS PASSED
  Phase 1: 5677 complete, 6323 pruned, best=0.0043
  Deduped: 75 -> 75 unique configs
  Best: trial 4603  robust=-0.0774  PnL=0.48%  trades=2005  (34.5min)
  Stress diag: evaluated=75 pruned=69 cache_hits=0 misses=75
  Recent 28d: FAIL — recent objective score -0.0062 <= 0; recent PnL -0.1877% < 0
  Holdout: PASS
  Sensitivity: penalty=0.6429
  Clustering: CLUSTERED
  Parity: short=N/A, long=N/A
  Total time: (34.5min)
  EXPORTED  yaml=artifacts/sweep/mexc/ADA-USDT_5m_screening_best.yaml  checks=SOME FAIL

════════════════════════════════════════════════════════════
  [2/56] mexc / APT-USDT / 5m
════════════════════════════════════════════════════════════


Callbacks are not supported in parallel mode. They will be skipped. Use post-hoc analysis instead.


  Split: dev=35021 holdout=8755 recent=8065
  Candles: 51,841  Days: 180.0  WF: 42.0/14.0/14.0d  Ref: 1.7370
  Training window: 180d of 201d available (capped to 180d)
Preflight: ALL CHECKS PASSED
  Phase 1: 5409 complete, 6591 pruned, best=-0.0026
  Deduped: 75 -> 75 unique configs
  Best: trial 7381  robust=-0.0383  PnL=-1.17%  trades=730  (33.8min)
  Stress diag: evaluated=75 pruned=69 cache_hits=0 misses=75
  Recent 28d: FAIL — recent objective score -0.0045 <= 0
  Holdout: FAIL
  Sensitivity: penalty=0.0000
  Clustering: CLUSTERED
  Parity: short=N/A, long=N/A
  Total time: (33.8min)
  EXPORTED  yaml=artifacts/sweep/mexc/APT-USDT_5m_screening_best.yaml  checks=SOME FAIL

════════════════════════════════════════════════════════════
  [3/56] mexc / ASTER-USDT / 5m
════════════════════════════════════════════════════════════


...

Callbacks are not supported in parallel mode. They will be skipped. Use post-hoc analysis instead.


  Split: dev=35194 holdout=8798 recent=8065
  Candles: 52,057  Days: 180.8  WF: 42.0/14.0/14.0d  Ref: 1.6630
  Training window: 180d of 184d available (capped to 180d)
Preflight: ALL CHECKS PASSED
  Phase 1: 5744 complete, 6256 pruned, best=-0.0003
  Deduped: 75 -> 75 unique configs
  Best: trial 7446  robust=-0.0277  PnL=1.39%  trades=979  (33.3min)
  Stress diag: evaluated=75 pruned=72 cache_hits=0 misses=75
  Recent 28d: PASS — 
  Holdout: PASS
  Sensitivity: penalty=0.0714
  Clustering: CLUSTERED
  Parity: short=N/A, long=N/A
  Total time: (33.3min)
  EXPORTED  yaml=artifacts/sweep/mexc/ZRO-USDT_5m_screening_best.yaml  checks=SOME FAIL

════════════════════════════════════════════════════════════
  [32/56] nonkyc / ALIAS-XMR / 5m
════════════════════════════════════════════════════════════
  SKIP: audit failed — ['total forward-fill fraction 0.5162 exceeds threshold 0.25', 'unexpected forward-fill fraction 0.5162 exceeds threshold 0.25']

═══════════════════════════════════════════

Callbacks are not supported in parallel mode. They will be skipped. Use post-hoc analysis instead.


  Split: dev=35194 holdout=8798 recent=8065
  Candles: 52,057  Days: 180.8  WF: 42.0/14.0/14.0d  Ref: 0.2015
  Training window: 180d of 184d available (capped to 180d)
Preflight: ALL CHECKS PASSED


Bar 15188: no orders placed (this warning will not repeat)
Bar 14921: no orders placed (this warning will not repeat)
Bar 20116: no orders placed (this warning will not repeat)
Bar 23084: no orders placed (this warning will not repeat)


## 4. Results Summary

In [ ]:
# Print discovery exclusion stats
if stale_exclusions:
    print(f"Stale pairs excluded : {len(stale_exclusions)}")
if insufficient_exclusions:
    print(f"Insufficient data    : {len(insufficient_exclusions)}")
print()

# Build summary table
summary_rows = []
for r in sweep_results:
    row = {
        "Exchange": r["connector"],
        "Pair": r["pair"],
        "Interval": r.get("interval", "—"),
        "Status": r["status"],
    }
    if r["status"] == "complete":
        row.update({
            "Robust": f"{r['robust_score']:.2f}",
            "PnL%": f"{r['pnl_pct']:.2f}",
            "Sharpe": f"{r['sharpe']:.2f}",
            "MaxDD%": f"{r['max_dd_pct']:.2f}",
            "Trades": r["trade_count"],
            "PF": f"{r['profit_factor']:.2f}" if r["profit_factor"] != float('inf') else "∞",
            "Fees": f"{r['total_fees']:.2f}",
            "WorstStress": r.get("worst_scenario", ""),
            "Exported": "✓" if r.get("exported") else "✗",
            "Checks": "PASS" if r.get("all_checks_pass") else "—",
        })
    else:
        row.update({k: "—" for k in ["Robust", "PnL%", "Sharpe", "MaxDD%", "Trades",
                                       "PF", "Fees", "WorstStress", "Exported", "Checks"]})
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

# Sort within each connector: completed + exported first, then by robust score
def sort_key(row):
    robust = float(row["Robust"]) if row["Robust"] != "—" else 0.0
    if row["Status"] != "complete":
        return (row["Exchange"], 2, 0, row["Pair"])
    if row["Exported"] == "✓":
        return (row["Exchange"], 0, -robust, row["Pair"])
    return (row["Exchange"], 1, -robust, row["Pair"])

summary_df["_sort"] = summary_df.apply(sort_key, axis=1)
summary_df = summary_df.sort_values("_sort").drop(columns=["_sort"]).reset_index(drop=True)

overall_complete = len([r for r in sweep_results if r["status"] == "complete"])
overall_exported = len([r for r in sweep_results if r.get("exported")])
overall_profitable = len([r for r in sweep_results if r["status"] == "complete" and r["robust_score"] >= MIN_ROBUST_SCORE])

print(f"{'='*60}")
print(f"  CROSS-EXCHANGE SWEEP RESULTS")
print(f"{'='*60}\n")
print(f"  Total connector/pairs scanned : {len(candidates)}")
print(f"  Completed                     : {overall_complete}")
print(f"  Profitable                    : {overall_profitable} (robust score >= {MIN_ROBUST_SCORE})")
print(f"  Exported                      : {overall_exported}")
print()

for connector in CONNECTORS:
    interval = INTERVALS_BY_CONNECTOR.get(connector, DEFAULT_INTERVAL)
    connector_df = summary_df[summary_df["Exchange"] == connector].drop(columns=["Exchange"]).reset_index(drop=True)

    n_scanned = len([c for c in candidates if c["connector"] == connector])
    n_complete = len([r for r in sweep_results if r["connector"] == connector and r["status"] == "complete"])
    n_exported = len([r for r in sweep_results if r["connector"] == connector and r.get("exported")])
    n_profitable = len([r for r in sweep_results if r["connector"] == connector and r["status"] == "complete"
                        and r["robust_score"] >= MIN_ROBUST_SCORE])

    print(f"{'-'*60}")
    print(f"  {connector.upper()} / {QUOTE_ASSET} / {interval}")
    print(f"{'-'*60}")
    print(f"  Total pairs scanned : {n_scanned}")
    print(f"  Completed           : {n_complete}")
    print(f"  Profitable          : {n_profitable}")
    print(f"  Exported            : {n_exported}")
    print()

    if connector_df.empty:
        print("No results for this exchange.")
    else:
        display(connector_df)


## 5. Profitable Pairs Detail by Exchange

In [ ]:
profitable = [r for r in sweep_results if r["status"] == "complete"
              and r["robust_score"] is not None and r["robust_score"] >= MIN_ROBUST_SCORE]
profitable.sort(key=lambda r: (r["connector"], -r["robust_score"], r["pair"]))

if not profitable:
    print("No profitable pairs found in this sweep.")
    print(f"Try adjusting MIN_ROBUST_SCORE (currently {MIN_ROBUST_SCORE}) or running on different exchanges.")
else:
    for connector in CONNECTORS:
        connector_profitable = [r for r in profitable if r["connector"] == connector]
        if not connector_profitable:
            print(f"\n{'='*60}")
            print(f"  {connector.upper()}: no profitable pairs")
            print(f"{'='*60}")
            continue

        print(f"\n{'='*60}")
        print(f"  {connector.upper()} profitable pairs")
        print(f"{'='*60}")

        for i, r in enumerate(connector_profitable):
            print(f"\n{'─'*60}")
            print(f"  #{i+1}  {r['pair']}  (robust={r['robust_score']:.4f})")
            print(f"{'─'*60}")
            print(f"  PnL %         : {r['pnl_pct']:.4f}")
            print(f"  Sharpe        : {r['sharpe']:.4f}")
            print(f"  Max DD %      : {r['max_dd_pct']:.4f}")
            print(f"  Trades        : {r['trade_count']}")
            print(f"  Profit Fac.   : {r['profit_factor']:.4f}")
            print(f"  Fees          : {r['total_fees']:.4f}")
            print(f"  Worst stress  : {r['worst_scenario']} ({r['worst_score']:.4f})")
            print(f"  Amount (quote): {r['best_config'].total_amount_quote:.2f}  "
                  f"(search range: 25.00 – 1000.00)")
            print(f"  Data          : {r['n_candles']:,} candles, {r['dataset_days']:.1f} days")
            if r.get("yaml_path"):
                print(f"  YAML          : {r['yaml_path']}")
            print(f"  Checks        : {'ALL PASS' if r.get('all_checks_pass') else 'SOME FAILED'}")

        print(f"\nCheck artifacts/sweep/{connector}/ for configs and reports.")


## 6. Next Steps

For each exported pair:
1. **Review the report** in `artifacts/sweep/<connector>/`
2. **Verify stop-ship checks** and YAML validation results
3. **Paper trade** using Hummingbot's paper trading mode
4. **Monitor** for at least 1 week before live trading
5. **Compare** live performance to backtest expectations

To re-run for a different set of exchanges, edit `CONNECTORS` in the configuration cell and Run All.
